# 03. Earnings Analyst Agent (`EarningsAnalyst`) Demo

### Architecture Role & Responsibilities
The **Earnings Analyst** conducts quantitative fundamental equity analysis. It queries financial statements and performs retrieval over SEC filings using:

- `get_earnings_data`: Financial statements and quarterly ratios
- `search_sec_filings`: Semantic/keyword retrieval over SEC EDGAR filings

In [1]:
import sys
from pathlib import Path

# Resolve src directory
project_root = Path.cwd().parent if Path.cwd().name == 'demo' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from research_agent.agents.earnings_analyst import earnings_analyst_agent
from research_agent.interfaces import get_llm

llm = get_llm()
earnings_analyst = earnings_analyst_agent(llm=llm)
print(f"Agent: {earnings_analyst.name}")
print(f"Tools: {list(earnings_analyst.tools.keys())}")

Agent: EarningsAnalyst
Tools: ['get_earnings_data', 'search_sec_filings']


## Sending Fundamental Equity Analysis Request

In [2]:
prompt = "Analyze fundamental financial performance and SEC 10-Q filing disclosures for NVDA"
print(f"Sending Input: '{prompt}'\n")

resp = earnings_analyst.run(prompt)
print("--- EARNINGS ANALYST OUTPUT ---")
print(resp.content)

Sending Input: 'Analyze fundamental financial performance and SEC 10-Q filing disclosures for NVDA'

--- EARNINGS ANALYST OUTPUT ---
**NVIDIA Corp. (NASDAQ: NVDA) – Q3 FY 2026 (Quarter ended July 31 2026)**  

---

### 1. Core Financial Metrics (from live earnings data)

| Metric | Q3 FY 2026 | Comments |
|--------|------------|----------|
| **Revenue** | **$96.22 B** | ↑ 105.9 % YoY; ↑ 17.9 % QoQ【source】 |
| **Gross Profit** | $72.14 B | Gross margin **74.98 %**【source】 |
| **Operating Income** | $63.73 B | Operating margin **66.24 %**【source】 |
| **Net Income** | $59.69 B | Net margin **62.03 %**【source】 |
| **Diluted EPS (GAAP)** | $2.22 | Beat estimate $2.09 → **+6.2 %** surprise【source】 |
| **Free Cash Flow (annual)** | **$96.68 B** | Strong cash generation relative to earnings【source】 |
| **Trailing P/E** | 28.99x | Valuation still elevated vs historic ~20‑25x range |
| **Forward P/E** | 14.41x | Reflects market’s expectation of rapid earnings growth |
| **Market Capitalisation**

## Inspecting Tool Execution Trace

In [3]:
print("--- TOOL EXECUTION TRACE ---")
for step in resp.trace:
    if step.step_type == "tool_call":
        print(f"TOOL CALL   : {step.content.get('tool')}({step.content.get('arguments')})")
    elif step.step_type == "tool_result":
        preview = str(step.content)[:90].replace('\n', ' ')
        print(f"OBSERVATION : {preview}...")
    elif step.step_type == "final_answer":
        print(f"FINAL ANSWER: Generated ({len(str(step.content))} chars)")

--- TOOL EXECUTION TRACE ---
TOOL CALL   : get_earnings_data({'ticker': 'NVDA'})
OBSERVATION : {   "ticker": "NVDA",   "quarter_end": "2026-07-31",   "revenue": "$96.22B",   "revenue_gr...
TOOL CALL   : search_sec_filings({'form_type': '10-Q', 'ticker': 'NVDA', 'query': 'segment revenue growth', 'top_k': 5})
OBSERVATION : {   "ticker": "NVDA",   "form_type": "10-Q",   "filing_date": "2026-08-26",   "search_meth...
TOOL CALL   : search_sec_filings({'form_type': '10-Q', 'query': 'balance sheet risk', 'ticker': 'NVDA', 'top_k': 5})
OBSERVATION : {   "ticker": "NVDA",   "form_type": "10-Q",   "filing_date": "2026-08-26",   "search_meth...
FINAL ANSWER: Generated (6925 chars)
